In [1]:
import sys
import os
from importlib.metadata import version, PackageNotFoundError

print("Python:", sys.version)
print("Working directory:", os.getcwd())

packages = [
    "openai",
    "streamlit",
    "langchain",
    "langchain-openai",
    "langgraph",
    "pydantic",
    "sqlalchemy",
    "pymysql",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: not installed")

print("\nRelevant environment variable names (values hidden):")
for name in sorted(os.environ):
    if any(word in name.upper() for word in ["OPENAI", "AZURE", "PROXY"]):
        print(name)

Python: 3.10.2 (main, Mar  4 2022, 02:09:05) [GCC 9.3.0]
Working directory: /voc/work
openai: 0.28.1
streamlit: 1.28.2
langchain: 0.0.335
langchain-openai: not installed
langgraph: not installed
pydantic: 2.5.2
sqlalchemy: 1.4.20
pymysql: 1.0.2

Relevant environment variable names (values hidden):
OPENAI_API_BASE
OPENAI_API_KEY
OPENAI_BASE_URL
OPENAI_MODELS
VOC_PROXY_ROUTE_jupyter_8888


In [2]:
import os
import json
from urllib.parse import urlsplit

print("Configured models:")
raw_models = os.getenv("OPENAI_MODELS", "")

try:
    print(json.dumps(json.loads(raw_models), indent=2))
except (json.JSONDecodeError, TypeError):
    print(raw_models or "(not configured)")

print("\nAPI endpoint configuration:")
for name in ["OPENAI_API_BASE", "OPENAI_BASE_URL"]:
    value = os.getenv(name)
    if value:
        parsed = urlsplit(value)
        # Excludes credentials, query strings, and fragments.
        print(f"{name}: {parsed.scheme}://{parsed.hostname}{parsed.path}")
    else:
        print(f"{name}: not configured")

print("\nAPI key present:", bool(os.getenv("OPENAI_API_KEY")))

Configured models:
gpt-3.5-turbo,gpt-3.5-turbo-0125,gpt-3.5-turbo-1106,gpt-3.5-turbo-instruct,gpt-4-turbo,gpt-4-turbo-2024-04-09,gpt-4o-2024-05-13,gpt-4,gpt-4-0613,gpt-4-1106-preview,davinci-002,babbage-002,gpt-3.5-turbo-16k,gpt-4o,text-embedding-3-large,text-embedding-3-small,text-embedding-ada-002,gpt-4o-mini-2024-07-18,gpt-4o-mini,gpt-4o-2024-08-06,o1-2024-12-17,o1,o3-mini,o4-mini-2025-04-16,o4-mini,o3-2025-04-16,o3,gpt-4.1-nano-2025-04-14,gpt-4.1-mini-2025-04-14,gpt-4.1-mini,gpt-4.1-2025-04-14,gpt-4.1,gpt-5-mini,gpt-5,gpt-4.1-nano,gpt-5.2,gpt-5.1,gpt-5.4,gpt-5.4-pro,gpt-5.3-codex,gpt-5-nano,gpt-5.4-mini,gpt-5.4-nano,gpt-5.5,gpt-5.5-pro,gpt-image-2,gpt-image-1.5,gpt-image-1,gpt-image-1-mini,chatgpt-image-latest,gpt-5.6-terra,gpt-5.6-luna,gpt-5.6-sol,gpt-5.2-pro,gpt-5-pro,gpt-6-astra

API endpoint configuration:
OPENAI_API_BASE: https://openai.vocareum.com/v1
OPENAI_BASE_URL: https://openai.vocareum.com/v1

API key present: True


In [3]:
import os
import time
import openai

# Use the credentials and gateway supplied by the lab.
openai.api_key = os.environ["OPENAI_API_KEY"]
openai.api_base = (
    os.getenv("OPENAI_API_BASE")
    or os.environ["OPENAI_BASE_URL"]
)

started = time.perf_counter()

try:
    response = openai.ChatCompletion.create(
        model="gpt-4o-mini",
        messages=[
            {
                "role": "system",
                "content": "Follow the user's instruction exactly.",
            },
            {
                "role": "user",
                "content": "Reply with exactly: Banking assistant connection successful.",
            },
        ],
        temperature=0,
        max_tokens=20,
        request_timeout=30,
    )

    print("Reply:", response["choices"][0]["message"]["content"])
    print("Model:", response.get("model"))
    print("Usage:", dict(response.get("usage", {})))
    print("Elapsed seconds:", round(time.perf_counter() - started, 2))

except Exception as exc:
    print("Error type:", type(exc).__name__)

    # Redact the key if an error happens to include it.
    message = str(exc).replace(openai.api_key, "[REDACTED]")
    print("Error:", message)

Reply: Banking assistant connection successful.
Model: gpt-4o-mini-2024-07-18
Usage: {'prompt_tokens': 26, 'completion_tokens': 6, 'total_tokens': 32, 'prompt_tokens_details': <OpenAIObject at 0x7efd21c54c70> JSON: {
  "cached_tokens": 0,
  "audio_tokens": 0
}, 'completion_tokens_details': <OpenAIObject at 0x7efd21c54d10> JSON: {
  "reasoning_tokens": 0,
  "audio_tokens": 0,
  "accepted_prediction_tokens": 0,
  "rejected_prediction_tokens": 0
}}
Elapsed seconds: 0.91


In [4]:
from pathlib import Path

PROJECT_ROOT = Path("/voc/work/banking_support")

folders = [
    "agents",
    "tools",
    "database",
    "evaluation",
    "tests",
    "data",
    "docs",
]

for folder in folders:
    (PROJECT_ROOT / folder).mkdir(parents=True, exist_ok=True)

# Make these folders importable Python packages.
for folder in ["agents", "tools", "database", "evaluation", "tests"]:
    (PROJECT_ROOT / folder / "__init__.py").touch(exist_ok=True)

# Write only when a file does not already exist.
initial_files = {
    ".gitignore": (
        ".env\n"
        ".venv/\n"
        "__pycache__/\n"
        ".ipynb_checkpoints/\n"
        "*.pyc\n"
        "data/*.db\n"
        "data/*.db-*\n"
        "data/logs/\n"
    ),
    "README.md": (
        "# Banking Customer Support AI Assistant\n\n"
        "Applied Generative AI capstone project.\n\n"
        "Features: message classification, feedback handling, "
        "ticket creation, ticket-status retrieval, and evaluation.\n"
    ),
}

for filename, content in initial_files.items():
    path = PROJECT_ROOT / filename
    if not path.exists():
        path.write_text(content, encoding="utf-8")

print("Project created at:", PROJECT_ROOT)
for path in sorted(PROJECT_ROOT.iterdir()):
    print(f"  {path.name}{'/' if path.is_dir() else ''}")

Project created at: /voc/work/banking_support
  .gitignore
  README.md
  agents/
  data/
  database/
  docs/
  evaluation/
  tests/
  tools/


In [5]:
import shutil
import importlib.util
import socket

print("venv module available:",
      importlib.util.find_spec("venv") is not None)

for command in ["git", "mysql", "mysqld"]:
    print(f"{command}:", shutil.which(command) or "not found")

try:
    with socket.create_connection(("127.0.0.1", 3306), timeout=2):
        print("Local port 3306: accepting connections")
except OSError:
    print("Local port 3306: not reachable")

venv module available: True
git: /usr/bin/git
mysql: not found
mysqld: not found
Local port 3306: not reachable


In [2]:
!.venv/bin/pip install ipykernel
!.venv/bin/python -m ipykernel install --user --name=banking-env --display-name="Python (Banking)"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 33.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.8/831.8 kB 6.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 27.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 888.2/888.2 kB 6.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 768.4/768.4 kB 6.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24/24 [ipykernel]24 [ipykernel]ient]ne]
Installed kernelspec banking-env in /voc/work/.local/share/jupyter/kernels/banking-env


In [1]:
import sys
print(sys.executable)

/usr/local/bin/python3.10


In [2]:
!.venv/bin/python -m streamlit run app.py --server.address 0.0.0.0 --server.port 8501



2026-10-03 15:43:52.935 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.16.2.2:8501
  External URL: http://54.201.3.179:8501

^C
  Stopping...
